# Loadery dokumentów    

https://docs.langchain.com/oss/python/integrations/document_loaders#all-document-loaders

In [ ]:
from langchain_kreuzberg import KreuzbergLoader
from dotenv import load_dotenv
load_dotenv()

## Pliki .txt

### `langchain-community`

In [ ]:
from langchain_community.document_loaders import TextLoader

In [ ]:
loader = TextLoader("documents/AI.txt")  # AI_long.txt
loader

In [ ]:
documents = loader.load()
documents

In [ ]:
len(documents)

In [ ]:
document = documents[0]
document.metadata

In [ ]:
print(document.page_content)

In [ ]:
document.type

In [ ]:
document.model_dump()

### `KreuzbergLoader`

In [ ]:
documents = KreuzbergLoader(file_path="documents/AI.txt").load()
documents

In [ ]:
len(documents)

In [ ]:
document = documents[0]
document.metadata

In [ ]:
print(document.page_content)

In [ ]:
document.model_dump()

## Pliki .pdf

### `langchain-community`

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

**Single page**

In [ ]:
loader = PyMuPDFLoader("documents/AI.pdf")
loader

In [ ]:
documents = loader.load()
documents

In [ ]:
documents[0].page_content

In [ ]:
documents[0].metadata

---

**Multi page**

`PyMuPDFLoader` automatycznie dzieli plik pdf na dokumenty na podstawie stron.

In [ ]:
loader = PyMuPDFLoader("documents/AI_long.pdf")
documents = loader.load()

In [ ]:
len(documents)

In [ ]:
for doc in documents:
    print(doc.metadata, "\n")

In [ ]:
content = "\n\n=====\n\n".join([doc.page_content for doc in documents])

print(content)

### `KreuzbergLoader`

**Single page**

In [ ]:
documents = KreuzbergLoader(file_path="documents/AI.pdf").load()
documents

In [ ]:
documents[0].page_content

In [ ]:
documents[0].metadata

**Multi page**

In [ ]:
documents = KreuzbergLoader(file_path="documents/AI_long.pdf").load()

In [ ]:
len(documents)

Domyślnie `KreuzbergLoader` zwraca cały plik jako **jeden** dokument. Aby podzielić pdf na strony (jak `PyMuPDFLoader`), trzeba przekazać konfigurację z `PageConfig(extract_pages=True)`.

In [ ]:
from kreuzberg import ExtractionConfig, PageConfig

config = ExtractionConfig(pages=PageConfig(extract_pages=True))

documents = KreuzbergLoader(file_path="documents/AI_long.pdf", config=config).load()
len(documents)

In [ ]:
for doc in documents:
    print(doc.metadata, "\n")

In [ ]:
content = "\n\n=====\n\n".join([doc.page_content for doc in documents])

print(content)

## Pliki .docx

W przeciwieństwie do PDF-a plik DOCX nie ma stałego podziału na strony — zależy on od programu, czcionek i ustawień wydruku. Dlatego zarówno krótki, jak i długi plik są domyślnie zwracane jako jeden dokument.

### `langchain-community`

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

**Krótki dokument**

In [ ]:
loader = Docx2txtLoader("documents/AI.docx")

documents = loader.load()
documents

In [ ]:
documents[0].page_content

In [ ]:
documents[0].metadata

**Długi dokument**

In [ ]:
loader = Docx2txtLoader("documents/AI_long.docx")

documents = loader.load()
documents

In [ ]:
len(documents)

In [ ]:
documents[0].page_content

In [ ]:
documents[0].metadata

### `KreuzbergLoader`

**Krótki dokument**

In [ ]:
documents = KreuzbergLoader(file_path="documents/AI.docx").load()
documents

In [ ]:
documents[0].page_content

In [ ]:
documents[0].metadata

**Długi dokument**

In [ ]:
documents = KreuzbergLoader(file_path="documents/AI_long.docx").load()
documents

In [ ]:
len(documents)

In [ ]:
documents[0].page_content

In [ ]:
documents[0].metadata

> **ZADANIA**

## Własne loadery plików

Gotowe loadery nie są konieczne. Możemy stworzyć własne klasy dziedziczące po `BaseLoader` z `langchain-core`. Każda klasa powinna implementować metodę `lazy_load()`, która zwraca kolejne obiekty `Document`.

Do odczytania binarnych formatów PDF i DOCX nadal potrzebujemy bibliotek parsujących, ale sposób tworzenia dokumentów i metadanych definiujemy samodzielnie — bez loaderów z `langchain-community`.

In [ ]:
from pathlib import Path

import pymupdf
from docx import Document as DocxDocument

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

### Własny loader plików TXT

In [ ]:
class CustomTextLoader(BaseLoader):
    def __init__(self, file_path, encoding="utf-8", max_characters=None):
        self.file_path = Path(file_path)
        self.encoding = encoding

        if type(max_characters) != int and max_characters is not None:
            raise ValueError("max_characters must be None or integner")
            
        if max_characters is not None and max_characters <= 0:
            raise ValueError("max_characters must be greater than 0 or None")

        self.max_characters = max_characters

    def lazy_load(self):
        content = self.file_path.read_text(encoding=self.encoding)

        if self.max_characters is None:
            yield Document(
                page_content=content,
                metadata={"source": str(self.file_path), "context_length": len(content)},
            )
            return

        for chunk_index, start in enumerate(
            range(0, len(content), self.max_characters)
        ):
            page_content = content[start:start + self.max_characters]
            yield Document(
                page_content=page_content,
                metadata={
                    "source": str(self.file_path),
                    "chunk": chunk_index,
                    "start_index": start,
                    "context_length": len(page_content)
                },
            )

In [ ]:
loader = CustomTextLoader("documents/AI.txt")
documents = loader.load()
documents

In [ ]:
len(documents)

In [ ]:
document = documents[0]
document.metadata

In [ ]:
print(document.page_content)

---

In [ ]:
loader = CustomTextLoader(
    "documents/AI.txt",
    max_characters=1_000,
)
documents = loader.load()

len(documents)

In [ ]:
documents[0].page_content

In [ ]:
[(len(document.page_content), document.metadata) for document in documents]

In [ ]:
print(documents[0].page_content)

### Własny loader plików PDF

Podobnie jak `PyMuPDFLoader`, nasza klasa zwraca osobny dokument dla każdej strony PDF-a. Numery stron w metadanych zaczynają się od zera.

In [ ]:
class CustomPDFLoader(BaseLoader):
    def __init__(self, file_path):
        self.file_path = Path(file_path)

    def lazy_load(self):
        with pymupdf.open(self.file_path) as pdf:
            for page_number, page in enumerate(pdf):
                yield Document(
                    page_content=page.get_text(),
                    metadata={
                        "source": str(self.file_path),
                        "page": page_number,
                    },
                )

In [ ]:
loader = CustomPDFLoader("documents/AI_long.pdf")
documents = loader.load()
documents

In [ ]:
len(documents)

In [ ]:
for document in documents:
    print(document.metadata)

### Własny loader plików DOCX

In [ ]:
class CustomDocxLoader(BaseLoader):
    def __init__(self, file_path):
        self.file_path = Path(file_path)

    def lazy_load(self):
        docx = DocxDocument(self.file_path)
        paragraphs = [
            paragraph.text
            for paragraph in docx.paragraphs
            if paragraph.text.strip()
        ]

        yield Document(
            page_content="\n\n".join(paragraphs),
            metadata={"source": str(self.file_path)},
        )

In [ ]:
loader = CustomDocxLoader("documents/AI_long.docx")
documents = loader.load()
documents

In [ ]:
len(documents)

In [ ]:
document = documents[0]
document.metadata

In [ ]:
print(document.page_content)

## Loadery korzystające z API

### Dedykowany loader Wikipedii

`WikipediaLoader` pobiera artykuły przez publiczne API Wikipedii i zwraca je jako dokumenty LangChain. Nie wymaga klucza API. Jeden artykuł odpowiada jednemu obiektowi `Document`.

In [ ]:
from datetime import timedelta

import wikipedia
from langchain_community.document_loaders import WikipediaLoader

# Wikimedia wymaga opisowego User-Agenta z danymi kontaktowymi – dla generycznych limity są ostrzejsze (HTTP 429)
USER_AGENT = "RAG-training-notebook/1.0 (educational example; kontakt: TWÓJ_EMAIL)"

wikipedia.set_user_agent(USER_AGENT)
wikipedia.set_rate_limiting(True, min_wait=timedelta(seconds=1))

In [ ]:
loader = WikipediaLoader(
    query="sztuczna inteligencja",
    lang="pl",
    load_max_docs=3,
    doc_content_chars_max=10_000,
)
loader

In [ ]:
documents = loader.load()
documents

In [ ]:
len(documents)

In [ ]:
document = documents[0]
document.metadata

In [ ]:
print(document.page_content)

In [ ]:
document.model_dump()

### Uniwersalny loader API

Nie każde API ma gotowy loader w LangChain. Możemy przygotować własną klasę dziedziczącą po `BaseLoader`. Adres, parametry i nagłówki odpowiadają za połączenie z API, a przekazane funkcje określają, jak zamienić odpowiedź JSON na zawartość i metadane dokumentów.

In [ ]:
import json
import time

import requests

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

In [ ]:
class APILoader(BaseLoader):
    def __init__(
        self,
        url,
        params=None,
        headers=None,
        items_extractor=None,
        content_extractor=None,
        metadata_extractor=None,
    ):
        self.url = url
        self.params = params or {}
        self.headers = headers or {}
        self.items_extractor = items_extractor or (lambda data: [data])
        self.content_extractor = content_extractor or (
            lambda item: json.dumps(item, ensure_ascii=False, indent=2)
        )
        self.metadata_extractor = metadata_extractor or (lambda item: {})

    def lazy_load(self):
        for attempt in range(3):
            response = requests.get(
                self.url,
                params=self.params,
                headers=self.headers,
                timeout=30,
            )
            if response.status_code != 429:
                break
            # przy zbyt wielu requestach czekamy tyle, ile podaje serwer (Retry-After)
            time.sleep(int(response.headers.get("Retry-After", 2 ** (attempt + 1))))
        response.raise_for_status()
        data = response.json()

        for item in self.items_extractor(data):
            metadata = {
                "source": response.url,
                **self.metadata_extractor(item),
            }
            yield Document(
                page_content=self.content_extractor(item),
                metadata=metadata,
            )

Poniżej ten sam loader pobiera dane z API Wikipedii. Zmiana API wymaga jedynie podania innego adresu, parametrów oraz funkcji wyciągających zawartość i metadane z odpowiedzi JSON.

Używamy endpointu `action=query&prop=extracts` z parametrem `explaintext=1`, który zwraca **pełną treść artykułu** jako czysty tekst. Endpoint `rest_v1/page/summary` zwraca pod tym samym kluczem `extract` tylko pierwszy akapit.

In [ ]:
response = requests.get(
    "https://pl.wikipedia.org/w/api.php",
    params={
        "action": "query",
        "prop": "extracts|info",
        "explaintext": 1,
        "inprop": "url",
        "titles": "Sztuczna inteligencja",
        "redirects": 1,
        "format": "json",
        "formatversion": 2,
    },
    headers={"User-Agent": USER_AGENT},
    timeout=30,
)

In [ ]:
loader = APILoader(
    url="https://pl.wikipedia.org/w/api.php",
    params={
        "action": "query",
        "prop": "extracts|info",
        "explaintext": 1,
        "inprop": "url",
        "titles": "Sztuczna inteligencja",
        "redirects": 1,
        "format": "json",
        "formatversion": 2,
    },
    headers={"User-Agent": USER_AGENT},
    items_extractor=lambda data: data["query"]["pages"],
    content_extractor=lambda page: page["extract"],
    metadata_extractor=lambda page: {
        "title": page["title"],
        "source": page["fullurl"],
    },
)
loader

In [ ]:
documents = loader.load()
documents

In [ ]:
document = documents[0]
document.metadata

In [ ]:
print(document.page_content)

> **ZADANIA**